# 🔒 IEEE-CIS Fraud Detection — Data Preprocessing Pipeline

> **Goal:** Load raw CSVs, apply memory optimisation, feature engineering, and cleaning.
> **Output:** `data/processed_train.csv` (unbalanced) for EDA and splitting.


## ⚙️ Step 0 — Imports & Path Configuration

We import the standard scientific Python stack plus scikit-learn's base classes that allow us
to write custom, pipeline-compatible transformers.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR     = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR     = os.path.join(BASE_DIR, 'data')
EDA_CSV      = os.path.join(DATA_DIR, 'processed_train.csv')

print('Project root :', BASE_DIR)
print('Data dir     :', DATA_DIR)

Project root : /Users/hassan/Library/CloudStorage/OneDrive-HigherEducationCommission/ML-Project-hehe
Data dir     : /Users/hassan/Library/CloudStorage/OneDrive-HigherEducationCommission/ML-Project-hehe/data


## 🗜️ Step 1 — Memory Optimisation (`reduce_mem_usage`)

### What it does
Iterates every numeric column and downcasts it to the smallest dtype that can safely hold its
value range (e.g. `float64` → `float16`, `int64` → `int8`).

### Why it is necessary
The raw CSVs total **~1.5 GB**. Pandas defaults to 64-bit types for everything, often wasting
4× the RAM actually required. Downcasting typically achieves a **50–70 % memory reduction**,
making local experimentation feasible on a standard laptop without kernel crashes.

In [2]:
def reduce_mem_usage(df, verbose=True):
    """Automatically downcast numeric columns to the minimum safe dtype."""
    numerics  = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type.name in numerics:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                for t in [np.int8, np.int16, np.int32, np.int64]:
                    if np.iinfo(t).min < c_min and c_max < np.iinfo(t).max:
                        df[col] = df[col].astype(t); break
            else:
                for t in [np.float16, np.float32, np.float64]:
                    if np.finfo(t).min < c_min and c_max < np.finfo(t).max:
                        df[col] = df[col].astype(t); break
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f'  RAM: {start_mem:.1f} MB  →  {end_mem:.1f} MB  '
              f'({100*(start_mem-end_mem)/start_mem:.1f}% reduction)')
    return df

## 🔀 Step 2 — Data Merger (`DataMerger`)

### What it does
Performs a **left join** of the Transaction table onto the Identity table using `TransactionID`
as the join key. Also standardises identity column names (`id-01` → `id_01`).

### Why it is necessary
Fraud footprints rely heavily on device and network metadata stored only in the identity table.
Not every transaction has associated identity data — a left join preserves all transactions.
The dash/underscore mismatch between train and test identity columns would silently break
model inference at scoring time if left unfixed.

In [3]:
class DataMerger(BaseEstimator, TransformerMixin):
    """Left-join Transaction + Identity on 'TransactionID'; standardise column names."""
    def __init__(self, identity_df):
        self.identity_df = identity_df

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        print('  [DataMerger] Standardising names and merging ...')
        id_df = self.identity_df.copy()
        id_df.columns = [c.replace('-', '_') for c in id_df.columns]
        merged = X.merge(id_df, on='TransactionID', how='left')
        print(f'  Shape after merge: {merged.shape}')
        return merged

## ⏰ Step 3 — Temporal Feature Engineering (`TimeFeatureExtractor`)

### What it does
Derives **`Transaction_Hour`** (0–23) and **`Transaction_Day`** (0–6) from the raw
`TransactionDT` column (timedelta in seconds from an unknown reference datetime).

### Why it is necessary
Raw seconds form a monotonically increasing counter. Deriving hour-of-day and day-of-week
exposes temporal fraud signatures — e.g. card-testing spikes at 3 AM — with near-zero
computational cost.

In [4]:
class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extract hour-of-day and day-of-week from TransactionDT."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        print('  [TimeFeatureExtractor] Engineering temporal features ...')
        X = X.copy()
        X['Transaction_Hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_Day']  = (X['TransactionDT'] // 86400) % 7
        return X

In [5]:
class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p transformation to specified columns to reduce skewness."""
    def __init__(self, cols=['TransactionAmt']):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        print(f'  [LogTransformer] Applying log1p to {self.cols} ...')
        X = X.copy()
        for col in self.cols:
            if col in X.columns:
                X[col] = np.log1p(X[col])
        return X

## 🗑️ Step 4 — High-Null Column Pruning (`DropHighNulls`)

### What it does
During **`fit()`**, records columns exceeding the null threshold.
During **`transform()`**, drops those columns from any DataFrame passed in.

### Why it is necessary
Many `V`-series features have > 90 % missing values. Imputing them forces the model to
learn an imputed constant instead of a real signal, injecting noise and worsening the
"curse of dimensionality".

> **Threshold:** `0.70` — drop any column missing more than 70 % of its values.


In [6]:
class DropHighNulls(BaseEstimator, TransformerMixin):
    """Drop columns whose null fraction exceeds `threshold`."""
    def __init__(self, threshold=0.70):
        self.threshold = threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        null_frac = X.isnull().mean()
        self.cols_to_drop_ = null_frac[null_frac > self.threshold].index.tolist()
        print(f'  [DropHighNulls] {len(self.cols_to_drop_)} columns flagged '
              f'(>{self.threshold*100:.0f}% missing)')
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors='ignore')

## 🔡 Step 5 — Frequency Encoding (`FrequencyEncoder`)

### What it does
Replaces each category with its **normalised frequency** as learned from the training set.

### Why it is necessary
| Encoding | Problem |
|----------|---------|
| Label Encoding | Arbitrary integers mislead ordinal-sensitive algorithms |
| One-Hot Encoding | Hundreds of sparse binary columns — memory explosion |
| **Frequency Encoding** | One meaningful numeric per category — rare fraud fingerprints get small, distinctive values |

Fitted on training set only → **no data leakage** to the test set.

In [7]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Replace categories with their normalized frequency (fitted on training data)."""
    def __init__(self, cat_cols=None):
        self.cat_cols   = cat_cols
        self.freq_maps_ = {}

    def fit(self, X, y=None):
        cols = self.cat_cols or X.select_dtypes(include=['object','category']).columns.tolist()
        for col in cols:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(normalize=True, dropna=False).to_dict()
        print(f'  [FrequencyEncoder] Fitted on {len(self.freq_maps_)} columns')
        return self

    def transform(self, X):
        print(f'  [FrequencyEncoder] Encoding {len(self.freq_maps_)} columns ...')
        X = X.copy()
        for col, fmap in self.freq_maps_.items():
            if col in X.columns:
                X[col] = X[col].map(fmap).fillna(0)
        return X

## 🚀 Step 6 — Load Data & Run the Complete Pipeline


In [8]:
print('[1/5] Loading raw CSVs ...')
train_tx = pd.read_csv(os.path.join(DATA_DIR, 'train_transaction.csv'))
train_id = pd.read_csv(os.path.join(DATA_DIR, 'train_identity.csv'))
print(f'  train_transaction : {train_tx.shape}')
print(f'  train_identity    : {train_id.shape}')

[1/5] Loading raw CSVs ...


  train_transaction : (590540, 394)
  train_identity    : (144233, 41)


In [9]:
print('[2/5] Optimising memory ...')
train_tx = reduce_mem_usage(train_tx)
train_id = reduce_mem_usage(train_id)

[2/5] Optimising memory ...


  RAM: 1791.7 MB  →  558.9 MB  (68.8% reduction)
  RAM: 56.5 MB  →  37.3 MB  (34.1% reduction)


In [10]:
print('[3/5] Running preprocessing pipeline ...')
pipeline = Pipeline([
    ('merger',       DataMerger(identity_df=train_id)),
    ('time_extract', TimeFeatureExtractor()),
    ('log',           LogTransformer()),
    ('drop_nulls',   DropHighNulls(threshold=0.70)),
    ('freq_encoder', FrequencyEncoder()),
])
processed = pipeline.fit_transform(train_tx)

[3/5] Running preprocessing pipeline ...
  [DataMerger] Standardising names and merging ...


  Shape after merge: (590540, 434)
  [TimeFeatureExtractor] Engineering temporal features ...


  [LogTransformer] Applying log1p to ['TransactionAmt'] ...


  [DropHighNulls] 208 columns flagged (>70% missing)


  [FrequencyEncoder] Fitted on 13 columns
  [FrequencyEncoder] Encoding 13 columns ...


In [11]:
print('[4/5] Final numeric cast + NaN fill ...')
for col in processed.select_dtypes(include=['object']).columns:
    processed[col] = pd.to_numeric(processed[col], errors='coerce')
processed.fillna(processed.median(numeric_only=True), inplace=True)

print(f'  Processed shape  : {processed.shape}')
print(processed['isFraud'].value_counts())
processed.head(3)

[4/5] Final numeric cast + NaN fill ...


  Processed shape  : (590540, 228)
isFraud
0    569877
1     20663
Name: count, dtype: int64


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V314,V315,V316,V317,V318,V319,V320,V321,Transaction_Hour,Transaction_Day
0,2987000,0,86400,4.242188,0.744522,13926,361.0,150.0,0.011263,142.0,...,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0,1
1,2987001,0,86401,3.400391,0.744522,2755,404.0,150.0,0.320414,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
2,2987002,0,86469,4.093750,0.744522,4663,490.0,150.0,0.651551,166.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


In [12]:
print(f'[5/5] Saving EDA dataset → {EDA_CSV}')
processed.to_csv(EDA_CSV, index=False)
print('  ✅ processed_train.csv saved (unbalanced).')

[5/5] Saving EDA dataset → /Users/hassan/Library/CloudStorage/OneDrive-HigherEducationCommission/ML-Project-hehe/data/processed_train.csv


  ✅ processed_train.csv saved (unbalanced).
